# MobilityDuck on the BerlinMOD Hanoi data (Chapter 7)

Every SQL listing of Sect. "BerlinMOD Hanoi" of Chapter 7, in the order of the chapter. Set before starting Jupyter:

- `PG_HANOI`: the connection string of the Hanoi database that `load/hanoi_pg.sql` of this chapter loads (for example `dbname=hanoi`);
- `MOBILITYDUCK_EXTENSION`: the path of `mobilityduck.duckdb_extension`.

The notebook runs DuckDB v1.5.6, as the README of this chapter explains. Its session is in UTC, as the chapter states, and its first cell reads the tables of the Hanoi database into DuckDB with `load/hanoi_duckdb.sql`.

In [1]:
import os

import duckdb
import pandas as pd

PG_HANOI = os.environ['PG_HANOI']
MOBILITYDUCK_EXTENSION = os.environ['MOBILITYDUCK_EXTENSION']
print(duckdb.__version__)

# The listings create their tables as the chapter prints them, so every run starts
# from a new database file
for f in ('hanoi_ch7.duckdb', 'hanoi_ch7.duckdb.wal'):
    if os.path.exists(f):
        os.remove(f)
con = duckdb.connect('hanoi_ch7.duckdb', config={'allow_unsigned_extensions': 'true'})
con.execute('INSTALL spatial')
con.execute('LOAD spatial')
con.execute("SET TimeZone = 'UTC'")


def run(sql, rows=20):
    """Run the statements of a listing one by one and show the first rows of the
    answer of the last query among them, the one whose answer the chapter prints: a
    SELECT or an EXPLAIN, or a statement answering other columns than the lone status,
    Count or Success, of an INSERT, an UPDATE, a LOAD or a CREATE INDEX (a PIVOT is a
    CREATE that answers its table). Only these rows are fetched, so that an answer of
    millions of rows, such
    as that of SELECT * EXCLUDE (SOG) FROM AISInput, stays in DuckDB."""
    sql = (sql.replace("'${MOBILITYDUCK_EXTENSION}'", f"'{MOBILITYDUCK_EXTENSION}'")
              .replace("'${PG_HANOI}'", f"'{PG_HANOI}'"))
    shown = None
    for st in con.extract_statements(sql):
        rel = con.execute(st)
        cols = [d[0] for d in rel.description] if rel.description else []
        if cols and (st.type in (duckdb.StatementType.SELECT, duckdb.StatementType.EXPLAIN)
                     or cols not in (['Count'], ['Success'])):
            shown = pd.DataFrame(rel.fetchmany(rows), columns=cols)
    return shown

1.5.6


## Loading the Hanoi tables

The tables of the Hanoi database, read with `postgres_scan` by `load/hanoi_duckdb.sql`, and their sizes.

In [2]:
run("""
-- The Hanoi tables of chapter 7 in DuckDB with MobilityDuck, read from the PostgreSQL database
-- hanoi, ${PG_HANOI} (loaded by load/hanoi_pg.sql) with postgres_scan, as the chapter's
-- listing of the table Ships reads its rows.
INSTALL spatial;
INSTALL postgres;
LOAD spatial;
LOAD '${MOBILITYDUCK_EXTENSION}';
LOAD postgres;
CREATE OR REPLACE TABLE Trips AS
SELECT TripId, VehicleId, Trip::tgeompoint AS Trip
FROM postgres_scan('${PG_HANOI}', 'public', 'trips');
ALTER TABLE Trips ADD COLUMN Trajectory GEOMETRY;
UPDATE Trips SET Trajectory = trajectory(Trip);
CREATE OR REPLACE TABLE Municipalities AS
SELECT MunicipalityId, MunicipalityName, Population, PercPop, PopDensityKm2, NoEnterp,
  PercEnterp, MunicipalityGeo
FROM postgres_scan('${PG_HANOI}', 'public', 'municipalities');
CREATE OR REPLACE TABLE Points AS
SELECT PointId, PosX, PosY, ST_Point(PosX, PosY) AS Geom
FROM postgres_scan('${PG_HANOI}', 'public', 'points');
CREATE OR REPLACE TABLE Vehicles AS
SELECT * FROM postgres_scan('${PG_HANOI}', 'public', 'vehicles');
CREATE OR REPLACE TABLE Licences AS
SELECT * FROM postgres_scan('${PG_HANOI}', 'public', 'licences');
CREATE OR REPLACE VIEW Licences1 AS SELECT * FROM Licences ORDER BY LicenceId LIMIT 10;
SELECT (SELECT COUNT(*) FROM Trips) AS trips, (SELECT COUNT(*) FROM Municipalities) AS munis,
  (SELECT COUNT(*) FROM Points) AS points, (SELECT COUNT(*) FROM Vehicles) AS vehicles,
  (SELECT SUM(numInstants(Trip)) FROM Trips) AS instants,
  (SELECT round(SUM(ST_Area(MunicipalityGeo))) FROM Municipalities) AS area;
""")

,trips,munis,points,vehicles,instants,area
0,2903,30,100,200,12744742,3.867079e+09


## MobilityDuck: BerlinMOD Hanoi

The queries of Sect. "BerlinMOD Hanoi" on the trips of 200 vehicles in Hanoi from June 1 to June 5, 2020.

*Display the trajectories, together with the municipalities of Hanoi.*

In [3]:
run("""
SELECT * FROM Trips;
SELECT * FROM Municipalities;
""")

,municipalityid,municipalityname,population,percpop,popdensitykm2,noenterp,percenterp,municipalitygeo
0,1,Quận Ba Đình,223100,0.03,24224,3981,0.03,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x96\x01...
1,2,Quận Bắc Từ Liêm,359200,0.04,7938,6409,0.05,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00:\x01\x0...
2,3,Quận Cầu Giấy,294500,0.03,23788,5255,0.03,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x07\x01...
3,4,Quận Đống Đa,377900,0.04,37980,6743,0.04,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x0e\x01...
4,5,Quận Hà Đông,435500,0.05,8773,7771,0.05,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\xf8\x01...
5,6,Quận Hai Bà Trưng,293900,0.03,28645,5244,0.03,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x04\x01...
6,7,Quận Hoàn Kiếm,140200,0.02,26206,2502,0.02,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x9c\x00...
7,8,Quận Hoàng Mai,539800,0.06,13431,9632,0.06,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00L\x01\x0...
8,9,Quận Long Biên,342700,0.04,5703,6115,0.04,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x92\x00...
9,10,Quận Nam Từ Liêm,290500,0.03,9030,5184,0.03,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x99\x01...


*Show the trips that cross the highest number of municipalities.*

In [4]:
run("""
WITH TripInt AS (
  SELECT VehicleId, TripId, COUNT(DISTINCT m.MunicipalityId) AS IntersectCount
  FROM Trips t, Municipalities m
  WHERE eIntersects(trip, MunicipalityGeo)
  GROUP BY VehicleId, TripId )
SELECT t1.*, ST_AsText(t.trajectory) AS trajectory
FROM Trips t JOIN TripInt t1 ON (t.TripId = t1.TripId AND t.VehicleId = t1.VehicleId)
ORDER BY IntersectCount DESC LIMIT 1;
""")

,vehicleid,tripid,IntersectCount,trajectory
0,50,723,15,LINESTRING (11732611.262793232 2421944.1212779...


*Total distance traveled per municipality.*

In [5]:
run("""
SELECT m.MunicipalityName,
  round(SUM(length(atGeometry(t.trip, m.MunicipalityGeo))) / 1000, 3) AS TotalKm
FROM Trips t, Municipalities m
WHERE eIntersects(t.trip, m.MunicipalityGeo)
GROUP BY m.MunicipalityName;
""")

,municipalityname,TotalKm
0,Quận Tây Hồ,1581.721
1,Huyện Đông Anh,7258.031
2,Huyện Ba Vì,3315.678
3,Quận Thanh Xuân,1468.664
4,Huyện Thường Tín,4175.125
5,Huyện Ứng Hòa,1828.720
6,Huyện Mỹ Đức,757.427
7,Huyện Sóc Sơn,9871.984
8,Huyện Thanh Oai,3023.081
9,Quận Đống Đa,1142.458


*Pairs of vehicles that have ever been at less than ten meters from each other.*

In [6]:
run("""
SELECT DISTINCT t1.VehicleId AS VehicleId1, t1.TripId AS TripId1,
  ST_AsText(t1.Trajectory) AS Traj1, t2.VehicleId AS VehicleId2, t2.TripId AS TripId2,
  ST_AsText(t2.Trajectory) AS Traj2
FROM ( SELECT * FROM Trips t1 ORDER BY TripId LIMIT 1000 ) t1,
  ( SELECT * FROM Trips t2 ORDER BY TripId LIMIT 1000 ) t2
WHERE t1.VehicleId < t2.VehicleId AND eDwithin(t1.Trip, t2.Trip, 10.0)
ORDER BY t1.VehicleId, t2.VehicleId;
""")

,VehicleId1,TripId1,Traj1,VehicleId2,TripId2,Traj2
0,1,1,LINESTRING (11789271.43645363 2393578.96220106...,47,667,LINESTRING (11774629.31666281 2345298.33469794...
1,1,12,LINESTRING (11789271.43645363 2393578.96220106...,47,677,LINESTRING (11774629.31666281 2345298.33469794...
2,2,19,LINESTRING (11781386.888427772 2413237.8711826...,4,47,LINESTRING (11794789.476820556 2408402.9568739...
3,2,23,LINESTRING (11781386.888427772 2413237.8711826...,11,148,LINESTRING (11790923.729391575 2395350.4959106...
4,2,26,LINESTRING (11797562.823822485 2407275.5251562...,13,190,LINESTRING (11770414.671685783 2415292.5975465...
5,2,22,LINESTRING (11797562.823822485 2407275.5251562...,40,574,LINESTRING (11804152.202968808 2388173.1348445...
6,2,17,LINESTRING (11781386.888427772 2413237.8711826...,60,859,LINESTRING (11785061.622742504 2434741.1939610...
7,3,31,LINESTRING (11769977.397594001 2386600.8720881...,51,734,LINESTRING (11756734.085128643 2376022.4816250...
8,3,38,LINESTRING (11758186.849011293 2357755.7793820...,68,995,LINESTRING (11769465.138693215 2383861.8579555...
9,3,30,LINESTRING (11758186.849011293 2357755.7793820...,68,989,LINESTRING (11769465.138693215 2383861.8579555...


*For each trip and vehicle, list the points from table Points*

In [7]:
run("""
WITH TripsTraj AS ( SELECT TripId, VehicleId, Trajectory FROM Trips ),
PointTrips AS (
  SELECT p.PointId, t2.VehicleId, t2.TripId, t2.Distance
  FROM Points p, LATERAL (
    SELECT t1.VehicleId, t1.TripId, ST_Distance(p.geom, t1.Trajectory) AS Distance
    FROM TripsTraj t1
    ORDER BY Distance LIMIT 3 ) AS t2 )
SELECT t.TripId, t.VehicleId, p.PointId, pt.Distance
FROM Trips t JOIN PointTrips pt ON (t.VehicleId = pt.VehicleId AND
  t.TripId = pt.TripId) JOIN Points p ON p.PointId = pt.PointId
ORDER BY t.VehicleId, t.TripId, p.PointId;
""")

,tripid,vehicleid,pointid,Distance
0,2,1,3,7.366692e+02
1,2,1,19,0.000000e+00
2,2,1,85,1.842773e+02
3,3,1,5,1.225403e+03
4,3,1,83,1.663583e+02
5,7,1,3,7.366692e+02
6,7,1,19,0.000000e+00
7,11,1,3,7.366692e+02
8,11,1,19,0.000000e+00
9,13,1,85,1.842773e+02


*Consider 30-minutes intervals between 2020-06-01 00:00:00 and 2020-06-02 23:30:00. How many vehicles were active per municipality and per interval?*

In [8]:
run("""
CREATE TABLE Bins(PId,SpanP) AS
WITH Intervals AS (
  SELECT ROW_NUMBER() OVER() AS pid, h AS StartTime,
    h + INTERVAL '30 minutes' AS EndTime
  FROM (
    SELECT generate_series AS h
    FROM generate_series(TIMESTAMPTZ '2020-06-01 00:00:00',
      TIMESTAMPTZ '2020-06-02 23:30:00', INTERVAL '30 minutes') ) AS t )
SELECT Pid, span(StartTime::timestamptz,EndTime::timestamptz) AS SpanP
FROM Intervals;

WITH TripPeriod AS (
  SELECT TripId, pId, p.SpanP, VehicleId, atTime(t.Trip, p.SpanP) AS tripPeriod
  FROM Trips t, Bins p
  WHERE t.Trip && p.SpanP)
SELECT t.Pid AS IntervalId, t.SpanP AS IntervalTime, m.MunicipalityName AS
  Municipality, COUNT(*) AS TripsCount
FROM TripPeriod t, Municipalities m
WHERE eIntersects(TripPeriod, m.MunicipalityGeo)
GROUP BY t.Pid, t.SpanP, m.MunicipalityName
ORDER BY COUNT(*) DESC, m.MunicipalityName, t.Pid;
""")

,IntervalId,IntervalTime,Municipality,TripsCount
0,84,"b'\'""\x01\x00\x00\x00\x00\x00\x00v\x92\x07\x1c...",Quận Long Biên,27
1,37,"b'\'""\x01\x00\x00\x00\x00\x00\x00\xe8\x04U\x08...",Huyện Đông Anh,24
2,84,"b'\'""\x01\x00\x00\x00\x00\x00\x00v\x92\x07\x1c...",Huyện Đông Anh,23
3,84,"b'\'""\x01\x00\x00\x00\x00\x00\x00v\x92\x07\x1c...",Huyện Gia Lâm,22
4,67,"b'\'""\x01\x00\x00\x00\x00\x00\x00\x84\xab\xe7\...",Huyện Sóc Sơn,22
5,84,"b'\'""\x01\x00\x00\x00\x00\x00\x00v\x92\x07\x1c...",Huyện Sóc Sơn,22
6,19,"b'\'""\x01\x00\x00\x00\x00\x00\x00$\xd4\xc9\x00...",Huyện Đông Anh,22
7,36,"b'\'""\x01\x00\x00\x00\x00\x00\x00\x16\xbb\xe9\...",Huyện Đông Anh,22
8,36,"b'\'""\x01\x00\x00\x00\x00\x00\x00\x16\xbb\xe9\...",Quận Cầu Giấy,22
9,19,"b'\'""\x01\x00\x00\x00\x00\x00\x00$\xd4\xc9\x00...",Quận Long Biên,22


*When and where did the vehicles with license plate numbers from Licences1*

In [9]:
run("""
WITH Meetings AS (
  SELECT l1.Licence AS Licence1, t2.VehicleId AS Car2Id,
    whenTrue(tDwithin(t1.Trip, t2.Trip, 3.0)) AS Periods
  FROM Trips t1, Licences1 l1, Trips t2, Vehicles v
  WHERE t1.VehicleId = l1.VehicleId AND t2.VehicleId = v.VehicleId AND
    t1.VehicleId <> t2.VehicleId AND t2.Trip && expandSpace(t1.trip::STBOX, 3.0) )
SELECT Licence1, Car2Id, Periods FROM Meetings WHERE Periods IS NOT NULL;
""")

,Licence1,Car2Id,Periods
0,B-DG 42,48,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
1,B-DG 42,76,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
2,B-DG 42,18,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
3,B-DG 42,96,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
4,B-NU 11,40,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
5,B-NU 11,36,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
6,B-NU 11,15,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
7,B-NU 11,17,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
8,B-NU 11,60,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
9,B-NU 11,82,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."


In [10]:
run("""
WITH Meetings AS (
  SELECT l1.Licence AS Licence1,
    t2.VehicleId AS Car2Id,  whenTrue(tDwithin(t1.Trip, t2.Trip, 3.0)) AS Periods,
    atTime(t1.trip, whenTrue(tDwithin(t1.Trip, t2.Trip, 3.0))) AS tt1,
    atTime(t2.trip, whenTrue(tDwithin(t1.Trip, t2.Trip, 3.0))) AS tt2
  FROM Trips t1, Licences1 l1, Trips t2, Vehicles v
  WHERE t1.VehicleId = l1.VehicleId AND t2.VehicleId = v.VehicleId AND
    t1.VehicleId <> t2.VehicleId AND t2.Trip && expandSpace(t1.trip::STBOX, 3.0) )
SELECT Licence1, Car2Id, ST_AsText(trajectory(tt1)),
  ST_AsText(trajectory(tt2)), Periods
FROM Meetings
WHERE Periods IS NOT NULL;
""")

,Licence1,Car2Id,st_astext(trajectory(tt1)),st_astext(trajectory(tt2)),Periods
0,B-DG 42,48,LINESTRING (11787967.180396773 2430334.6500129...,LINESTRING (11787970.180280417 2430334.6800063...,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
1,B-DG 42,76,LINESTRING (11782650.315933945 2431440.0777253...,LINESTRING (11782652.312050566 2431437.8381780...,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
2,B-DG 42,18,LINESTRING (11792419.386247838 2432262.2931606...,LINESTRING (11792416.601631982 2432261.1768360...,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
3,B-DG 42,96,LINESTRING (11796811.240615966 2431634.4806903...,LINESTRING (11796811.963550186 2431637.3922867...,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
4,B-NU 11,40,LINESTRING (11782892.286858017 2414031.4745087...,LINESTRING (11782891.511799334 2414028.5763509...,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
5,B-NU 11,36,LINESTRING (11783633.319076808 2423899.7730393...,LINESTRING (11783635.044591645 2423902.2271496...,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
6,B-NU 11,15,LINESTRING (11781685.85343928 2432801.26793784...,LINESTRING (11781685.372922016 2432804.2292051...,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
7,B-NU 11,17,LINESTRING (11783045.092530344 2417183.4469670...,LINESTRING (11783044.95822303 2417186.44398101...,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
8,B-NU 11,60,LINESTRING (11782536.21577093 2406755.20726715...,LINESTRING (11782535.675218642 2406758.1581814...,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."
9,B-NU 11,82,LINESTRING (11784292.625133881 2425922.3149138...,LINESTRING (11784293.435839554 2425925.2033046...,"b'\x00\x01\x00\x00(\'""\x00\x01\x00\x00\x00\x01..."


*For each trip, list the three vehicles that have been closest to that vehicle, together with the distance.*

In [11]:
run("""
CREATE VIEW Trips200 AS SELECT * FROM Trips ORDER BY TripId LIMIT 200;

SELECT t1.TripId, t1.VehicleId AS VehId1, t3.VehicleId AS VehId2, t3.Distance
FROM Trips200 t1, LATERAL (
  SELECT t2.VehicleId, minValue(t1.Trip <-> t2.Trip) AS Distance
  FROM Trips200 t2
  WHERE t1.VehicleId < t2.VehicleId AND TimeSpan(t1.Trip) && TimeSpan(t2.Trip)
  ORDER BY Distance LIMIT 3 ) AS t3
ORDER BY t1.TripId, t1.VehicleId, t3.Distance, t3.VehicleId;
""")

,tripid,VehId1,VehId2,Distance
0,1,1,10,3077.365220
1,1,1,12,6258.225685
2,1,1,7,11261.912354
3,2,1,7,17.901657
4,2,1,3,5229.145502
5,2,1,11,6440.952011
6,3,1,7,9823.670474
7,3,1,6,33173.511211
8,3,1,7,54842.896855
9,4,1,7,19622.154331


*For each group of ten disjoint vehicles, list the point(s) from the table Points, having the minimum aggregated distance from the given group during the given period.*

In [12]:
run("""
WITH Groups AS (
  SELECT ((ROW_NUMBER() OVER (ORDER BY v.Vehicleid))-1)//10 + 1 AS
    GroupId, v.Vehicleid
  FROM Vehicles v ),
SumDistances AS (
  SELECT g.GroupId, p.PointId, SUM(ST_Distance(t.trajectory, p.Geom)) AS SumDist
  FROM Groups g, Points p, Trips t
  WHERE t.Vehicleid = g.Vehicleid
  GROUP BY g.GroupId, p.PointId )
SELECT s1.GroupId, s1.PointId, s1.SumDist
FROM SumDistances s1
WHERE s1.SumDist <= ALL (
  SELECT SumDist
  FROM SumDistances s2
  WHERE s1.GroupId = s2.GroupId )
ORDER BY s1.GroupId, s1.PointId;
""")

,GroupId,pointid,SumDist
0,1,73,2.234304e+06
1,2,9,2.612559e+06
2,3,22,2.353185e+06
3,4,83,1.792626e+06
4,5,95,2.041819e+06
5,6,64,2.750816e+06
6,7,9,2.339360e+06
7,8,9,2.216792e+06
8,9,9,1.658948e+06
9,10,73,1.870781e+06


*For each passenger vehicle trip, find all points such that the trip passes within 10 meters of them. Additionally, for each such point, return the trip, the vehicle, the point, the trip trajectory, and the exact time when the vehicle first reaches that point.*

In [13]:
run("""
SELECT DISTINCT t.TripId, v.Licence, p.PointId, ST_AsText(p.Geom),
  startTimestamp(atValue(t.Trip, p.Geom)) AS Instant,
  ST_AsText(t.Trajectory) AS Trajectory
FROM Trips t, Vehicles v, Points p
WHERE t.VehicleId = v.VehicleId AND v.VehicleType = 'passenger' AND
  eIntersects(t.Trip, ST_Buffer(p.Geom, 10));
""")

,tripid,licence,pointid,st_astext(p.Geom),Instant,Trajectory
0,472,B-ZE 32,34,POINT (11773297.891025128 2389015.548325456),NaT,LINESTRING (11773884.811908385 2388432.7382521...
1,773,B-UG 53,45,POINT (11766753.807779608 2430297.614875895),2020-06-04 09:38:41.500875+00:00,LINESTRING (11777121.459499044 2436735.3290979...
2,614,B-OC 43,19,POINT (11782108.917777007 2391620.4009276605),2020-06-01 23:25:33.614502+00:00,LINESTRING (11781139.547651181 2365928.5132613...
3,103,B-ZN 7,19,POINT (11782108.917777007 2391620.4009276605),2020-06-05 10:32:58.004342+00:00,LINESTRING (11726607.001154417 2382351.6721441...
4,209,B-KH 14,19,POINT (11782108.917777007 2391620.4009276605),2020-06-05 17:47:09.944797+00:00,LINESTRING (11766715.079728764 2402783.1006607...
5,376,B-DB 26,19,POINT (11782108.917777007 2391620.4009276605),2020-06-03 09:32:33.268703+00:00,LINESTRING (11751933.49887988 2374701.04255507...
6,704,B-LY 49,19,POINT (11782108.917777007 2391620.4009276605),2020-06-03 17:10:08.879631+00:00,LINESTRING (11776593.77171684 2388438.05529756...
7,1609,B-XN 113,19,POINT (11782108.917777007 2391620.4009276605),2020-06-03 10:16:55.973207+00:00,LINESTRING (11773475.735043617 2371941.6130544...
8,1472,B-VQ 103,19,POINT (11782108.917777007 2391620.4009276605),2020-06-02 10:50:45.944876+00:00,LINESTRING (11783059.330193553 2434438.8833779...
9,55,B-CK 4,4,POINT (11794539.687015165 2408186.062678846),2020-06-05 17:21:43.966863+00:00,LINESTRING (11794789.476820556 2408402.9568739...


*Compute the first time a point is reached by any passenger vehicle.*

In [14]:
run("""
WITH Timestamps AS (
  SELECT DISTINCT  v.Licence, p.pointId, p.Geom,
    MIN(startTimestamp(atValue(t.Trip, p.Geom))) AS Instant
  FROM Trips t, Vehicles v, Points p
  WHERE t.VehicleId = v.VehicleId AND v.VehicleType = 'passenger' AND
    eIntersects(t.Trip, p.Geom)
  GROUP BY v.Licence, p.pointId, p.Geom )
SELECT t1.*
FROM Timestamps t1
WHERE t1.Instant <= ALL (
  SELECT t2.Instant
  FROM Timestamps t2
  WHERE t1.PointId = t2.PointId )
ORDER BY t1.PointId, t1.Licence;
""")

,licence,pointid,Geom,Instant
0,B-XN 113,19,b'\x01\x01\x00\x00\x00\xe3m^\x9d\xfbxfA\xfb\x9...,2020-06-01 08:49:20.668565+00:00
1,B-DO 179,30,b'\x01\x01\x00\x00\x00W\xfal\xe63lfA\x05\x041\...,2020-06-01 10:33:20.203525+00:00
2,B-UG 53,45,b'\x01\x01\x00\x00\x00\x9fT\xd99|qfA\xda@\xb4\...,2020-06-01 10:03:15.693069+00:00
3,B-VD 197,74,b'\x01\x01\x00\x00\x00\\\xffoU\xa4\x83fAou+\x8...,2020-06-01 09:56:53.377030+00:00
4,B-KH 14,83,b'\x01\x01\x00\x00\x00?&\xa0\x98\xccyfAe\xc6\x...,2020-06-02 22:38:29.382108+00:00
5,B-RO 69,88,b'\x01\x01\x00\x00\x00p\xa8#\xeb5~fA4\xc3\x8f\...,2020-06-02 20:48:29.908057+00:00
6,B-BW 192,90,b'\x01\x01\x00\x00\x00%x\xbe`\x92\x83fA\xc3`\x...,2020-06-01 19:21:07.830984+00:00
